Faiss

Facebook AI Similarity Search(Faiss) is a library for efficent for smilarity search and clustring of dense vectors. it contains algorithms that search in sets of vectors of ant size, up to ones that possibly do not fit in RAM. it also contains code for evaluation and parameter tuning.

In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import OllamaEmbeddings
from langchain_text_splitters import CharacterTextSplitter

loader=TextLoader("speech.txt")
documents=loader.load()
text_splitter=CharacterTextSplitter(chunk_size=1000,chunk_overlap=30)
docs=text_splitter.split_documents(documents)



In [7]:
docs

[Document(metadata={'source': 'speech.txt'}, page_content='The world must be made safe for democracy. Its peace must be planted upon the tested foundations of political liberty. We have no selfish ends to serve. We desire no conquest, no dominion. We seek no indemnities for ourselves, no material compensation for the sacrifices we shall freely make. We are but one of the champions of the rights of mankind. We shall be satisfied when those rights have been made as secure as the faith and the freedom of nations can make them.\n\nJust because we fight without rancor and without selfish object, seeking nothing for ourselves but what we shall wish to share with all free peoples, we shall, I feel confident, conduct our operations as belligerents without passion and ourselves observe with proud punctilio the principles of right and of fair play we profess to be fighting for.\n\n…'),
 Document(metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct our

In [9]:
embeddings=OllamaEmbeddings(model="gemma:2b")
db=FAISS.from_documents(docs, embeddings)
db

In [ ]:
###querying
query1="what does the speaker belives is the main reasons the United States should enter the war?"
query2="How does the speaker discribe the desire outcome of the war?"

docs=db.similarity_search(query2)
docs[0].page_content

'…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

## As a Retriever

we can also convert the vectorstore into a Retriever class. This allows us to esily use it in other Langchain methods, which largely work with retrievers

In [14]:
retriever=db.as_retriever()
docs= retriever.invoke(query2)
docs[0].page_content

'…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'

## Similarity search with score

there are some FAISS specific methods. One of them is similary_search_with_score, which allows you to return not only the documnets but also the distance score of the query to them. The returned distance score is L2 distance. Therefore, a lower score is better

In [15]:
docs_and_score= db.similarity_search_with_score(query2)
docs_and_score


[(Document(id='7b299b2e-6890-4f09-b6f9-c81b86b98a7f', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
  np.float32(2820.6821)),
 (Document(id='027e24d2-87ec-428d-aa37-d7c6f06b0d43', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed 

In [16]:
embedding_vector=embeddings.embed_query(query2)
embedding_vector

[-0.615519642829895,
 -1.7942262887954712,
 0.1732131540775299,
 1.3054213523864746,
 1.2462356090545654,
 -0.31162774562835693,
 -0.72835773229599,
 -0.03371374309062958,
 0.27932724356651306,
 -0.6198465824127197,
 0.9045524001121521,
 0.09860014915466309,
 -1.0388742685317993,
 1.4827641248703003,
 -1.4004873037338257,
 -1.0874171257019043,
 2.9274373054504395,
 1.2032783031463623,
 0.7581200003623962,
 1.0186654329299927,
 0.45010173320770264,
 -0.8110726475715637,
 0.04977879673242569,
 0.8456998467445374,
 0.6336076855659485,
 0.06418289989233017,
 -1.0946002006530762,
 -0.5518282651901245,
 -0.7833235263824463,
 -2.0840678215026855,
 -0.17349933087825775,
 -1.2227565050125122,
 0.48291003704071045,
 -0.42973092198371887,
 0.3110392391681671,
 -0.17176814377307892,
 2.0280849933624268,
 -0.07913459837436676,
 0.12050376087427139,
 -1.0779894590377808,
 -0.08384887129068375,
 0.8497216701507568,
 1.269506573677063,
 -1.7991900444030762,
 -1.5169093608856201,
 0.7505073547363281,
 

In [17]:
docs_score=db.similarity_search_by_vector(embedding_vector)
docs_score

[Document(id='7b299b2e-6890-4f09-b6f9-c81b86b98a7f', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='027e24d2-87ec-428d-aa37-d7c6f06b0d43', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. Ther

In [18]:
### saving and loading the vectorstore

db.save_local("faiss_index")

In [22]:
new_db = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
docs= new_db.similarity_search(query2)
docs

[Document(id='7b299b2e-6890-4f09-b6f9-c81b86b98a7f', metadata={'source': 'speech.txt'}, page_content='…\n\nIt will be all the easier for us to conduct ourselves as belligerents in a high spirit of right and fairness because we act without animus, not in enmity toward a people or with the desire to bring any injury or disadvantage upon them, but only in armed opposition to an irresponsible government which has thrown aside all considerations of humanity and of right and is running amuck. We are, let me say again, the sincere friends of the German people, and shall desire nothing so much as the early reestablishment of intimate relations of mutual advantage between us—however hard it may be for them, for the time being, to believe that this is spoken from our hearts.'),
 Document(id='027e24d2-87ec-428d-aa37-d7c6f06b0d43', metadata={'source': 'speech.txt'}, page_content='It is a distressing and oppressive duty, gentlemen of the Congress, which I have performed in thus addressing you. Ther